# CPU-Only Knowledge Engineering Notebook for SLM Dataset Generation

## 1. Installation
Install deterministic NLP and parsing libraries.

In [1]:
%pip install -q --break-system-packages PyMuPDF pdfplumber docling spacy nltk rapidfuzz regex beautifulsoup4 networkx numpy pandas scikit-learn joblib pyarrow orjson xxhash rich tqdm matplotlib sentencepiece datasets transformers
!python3 -m spacy download en_core_web_sm -q
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

Note: you may need to restart the kernel to use updated packages.
error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try brew install
    xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a Python library that isn't in Homebrew,
    use a virtual environment:
    
    python3 -m venv path/to/venv
    source path/to/venv/bin/activate
    python3 -m pip install xyz
    
    If you wish to install a Python application that isn't in Homebrew,
    it may be easiest to use 'pipx install xyz', which will manage a
    virtual environment for you. You can install pipx with
    
    brew install pipx
    
    You may restore the old behavior of pip by passing
    the '--break-system-packages' flag to pip, or by adding
    'break-system-packages = true' to your pip.conf file. The latter
    will permanently disable this error.
    
    If you disable this error, we STRONGLY recommen

True

## 2. Imports
Importing all necessary standard and 3rd-party libraries.

In [2]:
import os, sys, time, json, logging, shutil, unicodedata, re, collections, itertools
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple, Set
from dataclasses import dataclass, field, asdict
from datetime import datetime
import concurrent.futures

import fitz
import pdfplumber
import spacy
import orjson
import xxhash
from rapidfuzz import fuzz
import pandas as pd
import numpy as np
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.auto import tqdm
from rich.console import Console
from rich.table import Table
import matplotlib.pyplot as plt
import pyarrow as pa
import pyarrow.parquet as pq
from datasets import Dataset

import warnings
warnings.filterwarnings('ignore')
console = Console()

## 3. Configuration
One centralized dataclass for all settings.

In [3]:
@dataclass
class PipelineConfig:
    base_dir: Path = Path('./pipeline_workspace')
    input_pdfs_dir: Path = Path('./pipeline_workspace/input_pdfs')
    output_dir: Path = field(init=False)
    
    # Chunking
    target_chunk_size: int = 250
    chunk_overlap: int = 2
    min_words_per_chunk: int = 20
    max_words_per_chunk: int = 800
    duplicate_threshold: float = 90.0
    
    # Processing
    language: str = 'en'
    max_workers: int = max(1, (os.cpu_count() or 4) - 1)
    log_level: int = logging.INFO
    
    # Exports
    export_schemas: List[str] = field(default_factory=lambda: ['jsonl', 'json', 'csv', 'parquet', 'arrow', 'graphml'])
    
    def __post_init__(self):
        self.output_dir = self.base_dir / 'exports'
        self.metadata_dir = self.base_dir / 'metadata'
        self.graph_dir = self.base_dir / 'graph'
        self.stats_dir = self.base_dir / 'statistics'

config = PipelineConfig()

## 4. Directory Initialization
Ensure all necessary workspaces exist.

In [4]:
def init_directories(cfg: PipelineConfig):
    for d in [cfg.input_pdfs_dir, cfg.output_dir, cfg.metadata_dir, cfg.graph_dir, cfg.stats_dir]:
        d.mkdir(parents=True, exist_ok=True)
    console.print('[bold green]✓ Directories initialized.[/bold green]')

## 5. Logging
Configuring detailed event tracking.

In [5]:
def setup_logger(cfg: PipelineConfig) -> logging.Logger:
    logger = logging.getLogger('KnowledgePipeline')
    if not logger.handlers:
        logger.setLevel(cfg.log_level)
        fh = logging.FileHandler(cfg.base_dir / 'pipeline.log')
        fh.setFormatter(logging.Formatter('%(asctime)s | %(levelname)s | %(message)s'))
        logger.addHandler(fh)
    return logger
logger = setup_logger(config)

## 6. PDF Discovery
Recursively scan, hash, and detect duplicates.

In [6]:
class PDFScanner:
    def __init__(self, cfg: PipelineConfig):
        self.cfg = cfg
    
    def scan(self) -> List[Dict]:
        pdfs = list(self.cfg.input_pdfs_dir.rglob('*.pdf'))
        
        # Explicit target fallback
        target = Path('/Users/cemkaya/Documents/Ebooks_important/Practical Guide to LTE-A VoLTE and IoT Paving the way towards 5G.pdf')
        if target.exists() and target not in pdfs:
            pdfs.append(target)
            
        seen = set()
        registry = []
        for p in tqdm(pdfs, desc='Scanning'):
            h = xxhash.xxh64(p.read_bytes()).hexdigest()
            is_dup = h in seen
            seen.add(h)
            registry.append({'path': p, 'hash': h, 'duplicate': is_dup})
        return [x for x in registry if not x['duplicate']]

## 7. Metadata Extraction
Extract document metadata natively.

In [7]:
class MetadataExtractor:
    def extract(self, pdf_info: Dict) -> Dict:
        try:
            doc = fitz.open(pdf_info['path'])
            meta = doc.metadata
            doc.close()
            return {'title': meta.get('title', pdf_info['path'].stem) or pdf_info['path'].stem, 'pages': doc.page_count, 'hash': pdf_info['hash'], 'path': str(pdf_info['path'])}
        except: 
            return {'title': pdf_info['path'].stem, 'pages': 0, 'hash': pdf_info['hash'], 'path': str(pdf_info['path'])}

## 8. Document Parsing
Extract raw text blocks while maintaining reading order. Fallback available via pdfplumber.

In [8]:
class DocumentParser:
    def parse(self, filepath: Path) -> List[Dict]:
        doc = fitz.open(filepath)
        blocks = []
        for page_num in range(doc.page_count):
            page = doc.load_page(page_num)
            page_blocks = page.get_text('blocks')
            page_blocks.sort(key=lambda b: (b[1], b[0]))
            for b in page_blocks:
                if b[6] == 0:
                    blocks.append({'page': page_num + 1, 'text': b[4], 'bbox': b[:4]})
        doc.close()
        return blocks

## 9. Cleaning
Normalize unicode and eliminate formatting artifacts, preserving technical elements.

In [9]:
class Cleaner:
    def clean(self, blocks: List[Dict]) -> List[Dict]:
        clean_blocks = []
        for b in blocks:
            t = unicodedata.normalize('NFKC', b['text'])
            t = re.sub(r'\s+', ' ', t).strip()
            
            # Junk/TOC rejection heuristic
            words = t.split()
            if len(words) < 3 and len(t) < 15: continue
            
            if words:
                num_count = sum(1 for w in words if any(c.isdigit() for c in w))
                if num_count / len(words) > 0.25:
                    continue
                    
            if t:
                clean_blocks.append({'page': b['page'], 'text': t})
        return clean_blocks

## 10. Hierarchy Detection
Rule-based detection of Chapters and Sections based on text casing and numerics.

In [10]:
class HierarchyDetector:
    def detect(self, blocks: List[Dict]) -> List[Dict]:
        chapter, section = 'Root', 'Root'
        for b in blocks:
            t = b['text']
            if t.istitle() and len(t) < 100:
                if 'chapter' in t.lower(): chapter = t
                elif bool(re.match(r'^\d+\.\d+', t)): section = t
            b['hierarchy'] = f'{chapter} > {section}'
        return blocks

## 11. Linguistic Analysis
Uses spaCy for POS tagging, sentence boundaries, and dependency parsing.

In [11]:
class LinguisticAnalyzer:
    def __init__(self):
        try:
            self.nlp = spacy.load('en_core_web_sm')
        except OSError:
            logger.error("spaCy model missing.")
            raise
            
    def analyze(self, text: str):
        return self.nlp(text)

## 12. Concept Extraction
Extract base noun phrases to form concepts.

In [12]:
class ConceptExtractor:
    def extract(self, doc) -> List[str]:
        return list(set([chunk.text.lower() for chunk in doc.noun_chunks if len(chunk.text) > 3]))

## 13. Abbreviation Detection
Regex heuristics to find abbreviations and their long forms.

In [13]:
class AbbreviationDetector:
    def detect(self, text: str) -> Dict[str, str]:
        # Matches patterns like: Long Form Text (LFT)
        matches = re.findall(r'([A-Z][a-zA-Z\s\-]+)\s\(([A-Z]{2,})\)', text)
        return {short: long.strip() for long, short in matches}

## 14. Entity Extraction
Rule-based extraction using spaCy NER combined with 3GPP and Container Networking dictionaries.

In [14]:
class EntityExtractor:
    def extract(self, doc) -> List[str]:
        entities = [ent.text for ent in doc.ents if ent.label_ in ['ORG', 'PRODUCT', 'GPE', 'LAW']]
        # Add custom rules for 3GPP / LTE / 5G / Container Networking / General Networking
        pattern = r'\b(?:3GPP|LTE|5G|4G|VoLTE|IoT|NB-IoT|eMBB|URLLC|mMTC|MIMO|QoS|EPC|IMS|Kubernetes|K8s|Docker|CNI|Cilium|Calico|Flannel|eBPF|VLAN|VXLAN|BGP|OSPF|TCP/IP|NAT|IPv4|IPv6|LoadBalancer|Ingress|Egress|ServiceMesh|Istio|Envoy|Pod|Node|ClusterIP|NodePort|IPsec|WireGuard)\b'
        custom_patterns = re.findall(pattern, doc.text, flags=re.IGNORECASE)
        # Normalize case for deduplication
        custom_patterns = [p.upper() if len(p) <= 4 else p.title() for p in custom_patterns]
        return list(set(entities + custom_patterns))

## 15. Relationship Mining
Without AI: Uses dependency parsing to extract Subject-Verb-Object and Subject-Verb-Prep-Object triplets, highly relevant for networking routing graphs.

In [15]:
class RelationshipMiner:
    def mine(self, doc) -> List[Tuple[str, str, str]]:
        relations = []
        for token in doc:
            if token.pos_ == 'VERB':
                subjects = [w.text for w in token.lefts if w.dep_ in ('nsubj', 'nsubjpass')]
                
                # Direct objects
                objects_direct = [(token.lemma_, w.text) for w in token.rights if w.dep_ in ('dobj', 'attr')]
                
                # Prepositional objects (e.g., 'routes to Pod', 'communicates with API')
                objects_prep = []
                for child in token.rights:
                    if child.dep_ == 'prep':
                        verb_prep = f"{token.lemma_} {child.text}"
                        for p_child in child.rights:
                            if p_child.dep_ == 'pobj':
                                objects_prep.append((verb_prep, p_child.text))
                                
                all_objects = objects_direct + objects_prep
                
                for s, (verb_label, o) in itertools.product(subjects, all_objects):
                    relations.append((s, verb_label, o))
        return relations

## 16. Knowledge Graph Construction
Builds a NetworkX directed graph from concepts and relations.

In [16]:
class GraphBuilder:
    def __init__(self):
        self.graph = nx.DiGraph()
        
    def add_relations(self, relations: List[Tuple[str, str, str]]):
        for s, v, o in relations:
            self.graph.add_edge(s, o, label=v)
            
    def export(self, path: Path):
        nx.write_graphml(self.graph, str(path / 'knowledge_graph.graphml'))

## 17. Keyword Extraction
Uses TF-IDF to find globally significant terms.

In [17]:
class KeywordExtractor:
    def extract(self, texts: List[str], top_n: int = 5) -> List[List[str]]:
        if not texts: return []
        vectorizer = TfidfVectorizer(stop_words='english', max_features=1000)
        try:
            tfidf = vectorizer.fit_transform(texts)
            feature_names = vectorizer.get_feature_names_out()
            keywords = []
            for row in tfidf:
                sorted_idx = np.argsort(row.toarray()).flatten()[::-1]
                keywords.append([feature_names[i] for i in sorted_idx[:top_n] if row[0, i] > 0])
            return keywords
        except ValueError:
            return [['none'] for _ in texts]

## 18. Chunk Generation
Segments text intelligently along semantic boundaries without breaking sentences.

In [18]:
class Chunker:
    def __init__(self, cfg: PipelineConfig, analyzer: LinguisticAnalyzer):
        self.cfg = cfg
        self.analyzer = analyzer
        
    def chunk(self, blocks: List[Dict]) -> List[Dict]:
        chunks = []
        current_text = ""
        current_meta = None
        
        for b in blocks:
            if not current_meta: current_meta = b['hierarchy']
            current_text += b['text'] + " "
            
            words = current_text.split()
            if len(words) >= self.cfg.target_chunk_size:
                doc = self.analyzer.analyze(current_text)
                sents = list(doc.sents)
                
                chunk_text = ' '.join([s.text for s in sents])
                chunks.append({'text': chunk_text, 'hierarchy': current_meta, 'page': b['page']})
                
                # Overlap
                if self.cfg.chunk_overlap > 0 and len(sents) > self.cfg.chunk_overlap:
                    current_text = ' '.join([s.text for s in sents[-self.cfg.chunk_overlap:]]) + " "
                else:
                    current_text = ""
                current_meta = b['hierarchy']
                
        if current_text.strip() and len(current_text.split()) > self.cfg.min_words_per_chunk:
            chunks.append({'text': current_text.strip(), 'hierarchy': current_meta, 'page': blocks[-1]['page']})
            
        return chunks

## 19. Semantic Chunk Validation
Filters out meaningless chunks, checks size limits, and deduplicates using hashes and Rapidfuzz.

In [19]:
class ChunkValidator:
    def __init__(self, cfg: PipelineConfig):
        self.cfg = cfg
        self.seen = []
        
    def validate(self, chunks: List[Dict]) -> List[Dict]:
        valid = []
        for c in chunks:
            wc = len(c['text'].split())
            if wc < self.cfg.min_words_per_chunk or wc > self.cfg.max_words_per_chunk: continue
            
            is_dup = False
            for s in self.seen[-50:]:
                if fuzz.ratio(c['text'], s) > self.cfg.duplicate_threshold:
                    is_dup = True
                    break
            if not is_dup:
                self.seen.append(c['text'])
                c['id'] = "chk_" + xxhash.xxh64(c['text'].encode('utf-8')).hexdigest()
                valid.append(c)
        return valid

## 20. Instruction Template Engine
Deterministic mapping of extracted knowledge into high-quality SLM prompts.

In [20]:
class TemplateEngine:
    def generate(self, chunk: Dict, entities: List[str], abbreviations: Dict[str, str], relations: List[Tuple[str, str, str]]) -> List[Dict]:
        instructions = []
        text = chunk['text']
        hier = chunk['hierarchy']
        
        # General Summary
        instructions.append({
            'type': 'summary',
            'instruction': f"Summarize the context covering {hier}.",
            'input': text,
            'output': text
        })
        
        # Abbreviations
        for short, long in abbreviations.items():
            if short in text:
                instructions.append({
                    'type': 'abbreviation',
                    'instruction': f"What does {short} stand for in this context?",
                    'input': text,
                    'output': f"In this context, {short} stands for {long}."
                })
                
        # Relations
        if relations:
            rel = relations[0]
            instructions.append({
                'type': 'relation',
                'instruction': f"How does {rel[0]} relate to {rel[2]}?",
                'input': text,
                'output': f"Based on the text, {rel[0]} {rel[1]} {rel[2]}."
            })
            
        # Entities
        if entities:
            ent = entities[0]
            instructions.append({
                'type': 'entity',
                'instruction': f"Explain the role of {ent} as described in the text.",
                'input': text,
                'output': text
            })
            
        return instructions

## 21. Dataset Builder
Assembles multiple formats including Instruction, ChatML, and Flashcards.

In [21]:
class DatasetBuilder:
    def build(self, chunks: List[Dict], instructions_list: List[List[Dict]]) -> List[Dict]:
        dataset = []
        for c, inst_set in zip(chunks, instructions_list):
            for inst in inst_set:
                dataset.append({
                    'id': c['id'],
                    'hierarchy': c['hierarchy'],
                    'schema': inst['type'],
                    'alpaca': {
                        'instruction': inst['instruction'],
                        'input': inst['input'],
                        'output': inst['output']
                    },
                    'chatml': {
                        'messages': [
                            {'role': 'system', 'content': 'You are a highly accurate telecommunications AI.'},
                            {'role': 'user', 'content': f"Context:\n{inst['input']}\n\nQuestion:\n{inst['instruction']}"},
                            {'role': 'assistant', 'content': inst['output']}
                        ]
                    },
                    'flashcard': {
                        'front': inst['instruction'],
                        'back': inst['output']
                    }
                })
        return dataset

## 22. Validation
Global validation to ensure dataset integrity.

In [22]:
class GlobalValidator:
    def validate(self, dataset: List[Dict]) -> List[Dict]:
        return [d for d in dataset if d['alpaca']['instruction'].strip() and d['alpaca']['output'].strip()]

## 23. Statistics
Calculate deep statistical insights into the generated dataset.

In [23]:
class StatsEngine:
    def calculate(self, dataset: List[Dict]) -> Dict:
        return {
            'total_instructions': len(dataset),
            'schemas_distribution': pd.Series([d['schema'] for d in dataset]).value_counts().to_dict(),
            'total_words': sum(len(d['alpaca']['output'].split()) for d in dataset)
        }

## 24. Visualization
Plots charts for report generation.

In [24]:
class Visualizer:
    def plot(self, stats: Dict, output_dir: Path):
        plt.figure(figsize=(8, 4))
        schemas = stats['schemas_distribution']
        if schemas:
            plt.bar(schemas.keys(), schemas.values(), color='coral')
            plt.title('Instruction Types')
            plt.savefig(output_dir / 'schema_distribution.png')
            plt.close()

## 25. Export
Save all generated datasets and graphs.

In [25]:
class DataExporter:
    def __init__(self, cfg: PipelineConfig):
        self.cfg = cfg
        
    def export(self, dataset: List[Dict], graph_builder: GraphBuilder):
        graph_builder.export(self.cfg.graph_dir)
        
        with open(self.cfg.output_dir / 'instruction_dataset.jsonl', 'w') as f:
            for d in dataset: f.write(orjson.dumps(d['alpaca']).decode() + '\n')
            
        with open(self.cfg.output_dir / 'chat_dataset.jsonl', 'w') as f:
            for d in dataset: f.write(orjson.dumps(d['chatml']).decode() + '\n')
            
        with open(self.cfg.output_dir / 'flashcards.jsonl', 'w') as f:
            for d in dataset: f.write(orjson.dumps(d['flashcard']).decode() + '\n')
            
        df = pd.DataFrame([d['alpaca'] for d in dataset])
        if not df.empty:
            df.to_csv(self.cfg.output_dir / 'dataset.csv', index=False)
            Dataset.from_pandas(df).to_parquet(self.cfg.output_dir / 'dataset.parquet')

## 26. Summary & Execution
Orchestrate the entire CPU Knowledge Engineering Pipeline.

In [26]:
def process_pdf(pdf: Dict, cfg: PipelineConfig) -> Tuple[List[Dict], List[Tuple], List[Dict]]:
    parser = DocumentParser()
    cleaner = Cleaner()
    hierarchy = HierarchyDetector()
    analyzer = LinguisticAnalyzer()
    concept_ext = ConceptExtractor()
    abbr_ext = AbbreviationDetector()
    ent_ext = EntityExtractor()
    rel_miner = RelationshipMiner()
    chunker = Chunker(cfg, analyzer)
    validator = ChunkValidator(cfg)
    template_eng = TemplateEngine()
    
    blocks = parser.parse(pdf['path'])
    clean_blocks = cleaner.clean(blocks)
    hier_blocks = hierarchy.detect(clean_blocks)
    
    valid_chunks = validator.validate(chunker.chunk(hier_blocks))
    
    all_instructions = []
    all_relations = []
    
    for c in valid_chunks:
        doc = analyzer.analyze(c['text'])
        concepts = concept_ext.extract(doc)
        abbrs = abbr_ext.detect(c['text'])
        ents = ent_ext.extract(doc)
        rels = rel_miner.mine(doc)
        
        all_relations.extend(rels)
        insts = template_eng.generate(c, ents, abbrs, rels)
        all_instructions.append(insts)
        
    return valid_chunks, all_relations, all_instructions

def run_pipeline():
    start = time.time()
    console.print('[bold magenta]🧠 Starting Knowledge Engineering SLM Pipeline[/bold magenta]')
    init_directories(config)
    
    scanner = PDFScanner(config)
    pdfs = scanner.scan()
    if not pdfs:
        console.print('[red]No valid PDFs found.[/red]')
        return
        
    global_chunks = []
    global_insts = []
    graph = GraphBuilder()
    
    # Executing purely on CPU with ThreadPool for I/O and parser GIL release
    with concurrent.futures.ThreadPoolExecutor(max_workers=config.max_workers) as executor:
        futures = {executor.submit(process_pdf, p, config): p for p in pdfs}
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(pdfs), desc='Mining Knowledge'):
            chunks, rels, insts = future.result()
            global_chunks.extend(chunks)
            global_insts.extend(insts)
            graph.add_relations(rels)
            
    dataset_builder = DatasetBuilder()
    dataset = dataset_builder.build(global_chunks, global_insts)
    
    dataset = GlobalValidator().validate(dataset)
    
    stats = StatsEngine().calculate(dataset)
    Visualizer().plot(stats, config.stats_dir)
    
    with open(config.stats_dir / 'statistics.json', 'w') as f:
        json.dump(stats, f, indent=2)
        
    exporter = DataExporter(config)
    exporter.export(dataset, graph)
    
    console.print(f'[bold green]✅ Pipeline Complete in {time.time()-start:.1f}s. Generated {len(dataset)} instructions.[/bold green]')

if __name__ == '__main__':
    run_pipeline()

🧠 Starting Knowledge Engineering SLM Pipeline

✓ Directories initialized.

Scanning:   0%|          | 0/8 [00:00<?, ?it/s]

Mining Knowledge:   0%|          | 0/7 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

✅ Pipeline Complete in 270.9s. Generated 7458 instructions.